# CHARGEMENT DES DONNEES

In [ ]:
import string
import pandas as pd
import xlrd
from linearmodels.panel import PanelOLS

## CINEMA

Identifier les villes où il y a eu : fermeture, ouverture, statu quo, aucun cinéma --> 4 groupes

In [5]:
# URL du fichier Excel
url = "https://www.data.gouv.fr/api/1/datasets/r/cdb918e7-7f1a-44fc-bf6f-c59d1614ed6d"

# Charger les données
dfs = pd.read_excel(url, sheet_name=None)

# Afficher les premières lignes
print(dfs.keys())

dict_keys(['Sommaire', '2025', '2024', '2023', '2022', '2021', '2020', '2019', '2018', '2017', '2016', '2015', '2014', '2013', '2012', '2011', '2010', '2009', '2008', '2007', '2006', '2005', '2004', '2003', 'ESRI_MAPINFO_SHEET'])


In [6]:
dfs_final = []

for i in range(2003, 2026):
    df = dfs[str(i)]
    
    # La 4e ligne devient le nom des colonnes
    df.columns = df.iloc[3]
    
    # Supprimer les 4 premières lignes
    df = df.iloc[4:].reset_index(drop=True)

    # Nettoyer les noms de colonnes
    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
        .str.upper()
    )

    # Harmoniser les noms
    df = df.rename(columns={
        "N°AUTO": "n_auto",
        "NAUTOC": "n_auto",
        
        "NOMETAB": "nom_etablissement",
        "NOM ETABLISSEMENT": "nom_etablissement",

        "AE" : "art_et_essai",
        "ART ET ESSAI" : "art_et_essai",

        "ECRANS 3D" : "3d",
        "3D" : "3d",

        "CODE COMMUNE" : "code_commune",
        "COMMUNE" : "commune",
        "ECRANS" : "ecrans",
        "FAUTEUILS" : "fauteuils",
        "GENRE" : "genre",
    })
    
    # Ajouter l'année
    df["annee"] = i
    
    # Créer df_2003, df_2004, ..., df_2025
    globals()[f"df_{i}"] = df

    # Ajouter à la liste
    dfs_final.append(df)

# Empiler tous les DataFrames
df_cinema = pd.concat(dfs_final, ignore_index=True)

In [7]:
df_cinema.head()

3,n_auto,nom_etablissement,ecrans,fauteuils,code_commune,commune,genre,art_et_essai,annee,3d,DEPCOM
0,12,GEORGE V,11,1666,75108,Paris 8e Arrondissement,FIXE,NON,2003,NaN,NaN
1,31,UGC NORMANDIE,4,1610,75108,Paris 8e Arrondissement,FIXE,NON,2003,NaN,NaN
2,34,NaN,4,1017,75108,Paris 8e Arrondissement,FIXE,NON,2003,NaN,NaN
3,35,BALZAC,3,647,75108,Paris 8e Arrondissement,FIXE,A,2003,NaN,NaN
4,52,GAUMONT CHAMPS ELYSEES AMBASSADE,7,1564,75108,Paris 8e Arrondissement,FIXE,NON,2003,NaN,NaN


In [ ]:
df_cinema_fixe = df_cinema[df_cinema["genre"] == "FIXE"].copy()

# IDENTIFICATION DES COMMUNES DONT LE CINEMA A FERME

In [76]:
# Trier
df_cinema_fixe = df_cinema_fixe.sort_values(["n_auto", "annee"])

# Année suivante observée pour chaque cinéma
df_cinema_fixe["annee_suivante"] = (
    df_cinema_fixe.groupby("n_auto")["annee"].shift(-1)
)

# Une fermeture = dernière observation du cinéma,
# sauf si cette dernière observation est la dernière année du fichier
df_fermetures = df_cinema_fixe[
    (df_cinema_fixe["annee_suivante"].isna()) &
    (df_cinema_fixe["annee"] < df_cinema_fixe["annee"].max())
].copy()

# Garder les informations souhaitées
df_fermetures = df_fermetures[
    ["n_auto", "nom_etablissement", "code_commune", "annee"]
].rename(
    columns={"annee": "annee_fermeture"}
)

df_fermetures

3,n_auto,nom_etablissement,code_commune,annee_fermeture
34946,12,UGC GEORGE V,75108,2020
43137,31,UGC NORMANDIE,NaN,2024
8374,34,NaN,75108,2007
26774,52,GAUMONT CHAMPS ELYSEES AMBASSADE,75108,2016
41080,54,GAUMONT CHAMPS ELYSEES MARIGNAN,NaN,2023
...,...,...,...,...
6284,469492,NaN,43268,2005
43002,701401,LA TANIERE,NaN,2023
40651,715700,AUDITORIUM,73015,2022
42244,718210,LE CONCORDE,NaN,2023


# FIN TEST

In [62]:
df_cinema_agg = (
    df_cinema_fixe
    .groupby(["code_commune", "annee"], as_index=False)
    .agg(
        ecrans=("ecrans", "sum"),
        fauteuils=("fauteuils", "sum"),
        nb_cinemas=("n_auto", "nunique")
    )
)

In [63]:
df_cinema_agg = df_cinema_agg.sort_values(
    ["code_commune", "annee"]
)

df_cinema_agg["evol"] = (
    df_cinema_agg
    .groupby("code_commune")["nb_cinemas"]
    .diff()
    .fillna(0)
)

In [64]:
df_cinema_agg.head()

,code_commune,annee,ecrans,fauteuils,nb_cinemas,evol
0,01004,2003,1,238,1,0.0
1,01004,2004,1,238,1,0.0
2,01004,2005,4,833,2,1.0
3,01004,2006,3,595,1,-1.0
4,01004,2007,3,595,1,0.0


# LEGISLATIVES

# PRESIDENTIELLES

In [13]:
# Le jeu de données avec les résultats par candidat par bureau de vote
df_pres2022 = pd.read_excel("/Users/a33630/Documents/cinema/pres_2022.xlsx")
df_pres2017 = pd.read_excel("/Users/a33630/Documents/cinema/pres_2017.xls", engine='xlrd')
df_pres2012 = pd.read_excel("/Users/a33630/Documents/cinema/pres_2012.xls", engine='xlrd')
df_pres2007 = pd.read_excel("/Users/a33630/Documents/cinema/pres_2007.xls", engine='xlrd')

In [14]:
df_pres2022["code_commune"] = (
    df_pres2022["Code du département"].astype(str).str.zfill(2)
    + df_pres2022["Code de la commune"].astype(str).str.zfill(3)
)

df_pres2022["score_2022"] = 100*df_pres2022["Voix"] / df_pres2022["Exprimés"]

In [15]:
df_pres2012["code_commune"] = (
    df_pres2012["Code du département"].astype(str).str.zfill(2)
    + df_pres2012["Code de la commune"].astype(str).str.zfill(3)
)

df_pres2012["score_2012"] = 100*df_pres2012["Voix"] / df_pres2012["Exprimés"]

In [16]:
df_pres2007["code_commune"] = (
    df_pres2007["Code du département"].astype(str).str.zfill(2)
    + df_pres2007["Code de la commune"].astype(str).str.zfill(3)
)

df_pres2007["score_2007"] = 100*df_pres2007["Voix"] / df_pres2007["Exprimés"]

In [17]:
df_pres2017.head()

,Code du département,Libellé du département,Code de la commune,Libellé de la commune,Inscrits,Abstentions,% Abs/Ins,Votants,% Vot/Ins,Blancs,% Blancs/Ins,% Blancs/Vot,Nuls,% Nuls/Ins,% Nuls/Vot,Exprimés,% Exp/Ins,% Exp/Vot,N°Panneau,Sexe,Nom,Prénom,Voix,% Voix/Ins,% Voix/Exp,N°Panneau.1,Sexe.1,Nom.1,Prénom.1,Voix.1,% Voix/Ins.1,% Voix/Exp.1,N°Panneau.2,Sexe.2,Nom.2,Prénom.2,Voix.2,% Voix/Ins.2,% Voix/Exp.2,N°Panneau.3,...,Nom.5,Prénom.5,Voix.5,% Voix/Ins.5,% Voix/Exp.5,N°Panneau.6,Sexe.6,Nom.6,Prénom.6,Voix.6,% Voix/Ins.6,% Voix/Exp.6,N°Panneau.7,Sexe.7,Nom.7,Prénom.7,Voix.7,% Voix/Ins.7,% Voix/Exp.7,N°Panneau.8,Sexe.8,Nom.8,Prénom.8,Voix.8,% Voix/Ins.8,% Voix/Exp.8,N°Panneau.9,Sexe.9,Nom.9,Prénom.9,Voix.9,% Voix/Ins.9,% Voix/Exp.9,N°Panneau.10,Sexe.10,Nom.10,Prénom.10,Voix.10,% Voix/Ins.10,% Voix/Exp.10
0,1,Ain,1,L'Abergement-Clémenciat,598,92,15.38,506,84.62,2,0.33,0.40,9,1.51,1.78,495,82.78,97.83,2,F,LE PEN,Marine,126,21.07,25.45,3,M,MACRON,Emmanuel,119,19.90,24.04,11,M,FILLON,François,110,18.39,22.22,9,...,HAMON,Benoît,29,4.85,5.86,10,M,ASSELINEAU,François,6,1.00,1.21,5,F,ARTHAUD,Nathalie,4,0.67,0.81,6,M,POUTOU,Philippe,4,0.67,0.81,7,M,CHEMINADE,Jacques,2,0.33,0.40,8,M,LASSALLE,Jean,2,0.33,0.40
1,1,Ain,2,L'Abergement-de-Varey,209,25,11.96,184,88.04,6,2.87,3.26,2,0.96,1.09,176,84.21,95.65,2,F,LE PEN,Marine,48,22.97,27.27,3,M,MACRON,Emmanuel,37,17.70,21.02,11,M,FILLON,François,34,16.27,19.32,9,...,DUPONT-AIGNAN,Nicolas,6,2.87,3.41,5,F,ARTHAUD,Nathalie,2,0.96,1.14,6,M,POUTOU,Philippe,2,0.96,1.14,10,M,ASSELINEAU,François,1,0.48,0.57,7,M,CHEMINADE,Jacques,0,0.00,0.00,8,M,LASSALLE,Jean,0,0.00,0.00
2,1,Ain,4,Ambérieu-en-Bugey,8586,1962,22.85,6624,77.15,114,1.33,1.72,58,0.68,0.88,6452,75.15,97.40,2,F,LE PEN,Marine,1667,19.42,25.84,9,M,MÉLENCHON,Jean-Luc,1412,16.45,21.88,3,M,MACRON,Emmanuel,1332,15.51,20.64,11,...,HAMON,Benoît,344,4.01,5.33,6,M,POUTOU,Philippe,91,1.06,1.41,10,M,ASSELINEAU,François,71,0.83,1.10,8,M,LASSALLE,Jean,60,0.70,0.93,5,F,ARTHAUD,Nathalie,40,0.47,0.62,7,M,CHEMINADE,Jacques,5,0.06,0.08
3,1,Ain,5,Ambérieux-en-Dombes,1172,215,18.34,957,81.66,21,1.79,2.19,3,0.26,0.31,933,79.61,97.49,2,F,LE PEN,Marine,306,26.11,32.80,11,M,FILLON,François,197,16.81,21.11,3,M,MACRON,Emmanuel,191,16.30,20.47,9,...,HAMON,Benoît,37,3.16,3.97,6,M,POUTOU,Philippe,10,0.85,1.07,10,M,ASSELINEAU,François,10,0.85,1.07,8,M,LASSALLE,Jean,6,0.51,0.64,5,F,ARTHAUD,Nathalie,5,0.43,0.54,7,M,CHEMINADE,Jacques,0,0.00,0.00
4,1,Ain,6,Ambléon,99,20,20.20,79,79.80,2,2.02,2.53,0,0.00,0.00,77,77.78,97.47,9,M,MÉLENCHON,Jean-Luc,19,19.19,24.68,2,F,LE PEN,Marine,18,18.18,23.38,3,M,MACRON,Emmanuel,15,15.15,19.48,11,...,HAMON,Benoît,3,3.03,3.90,6,M,POUTOU,Philippe,2,2.02,2.60,5,F,ARTHAUD,Nathalie,1,1.01,1.30,8,M,LASSALLE,Jean,1,1.01,1.30,7,M,CHEMINADE,Jacques,0,0.00,0.00,10,M,ASSELINEAU,François,0,0.00,0.00


In [18]:
df_pres2017_lfn = df_pres2017.copy()

# Panneaux 0 à 10 : le panneau 0 correspond aux colonnes sans suffixe
for i in range(11):

    suffix = "" if i == 0 else f".{i}"

    # Si le panneau n°2 n'est pas présent, on met les colonnes à NaN
    mask = df_pres2017_lfn[f"N°Panneau{suffix}"] != 2

    cols = [
        f"N°Panneau{suffix}",
        f"Sexe{suffix}",
        f"Nom{suffix}",
        f"Prénom{suffix}",
        f"Voix{suffix}",
        f"% Voix/Ins{suffix}",
        f"% Voix/Exp{suffix}"
    ]

    df_pres2017_lfn.loc[mask, cols] = pd.NA


# Créer les colonnes finales
mapping = {
    "n_panneau": "N°Panneau",
    "sexe": "Sexe",
    "nom": "Nom",
    "prenom": "Prénom",
    "voix": "Voix",
    "voix_ins": "% Voix/Ins",
    "voix_exp": "% Voix/Exp"
}

for nouvelle_colonne, prefixe in mapping.items():

    cols = [
        f"{prefixe}" if i == 0 else f"{prefixe}.{i}"
        for i in range(11)
    ]

    df_pres2017_lfn[nouvelle_colonne] = (
        df_pres2017_lfn[cols]
        .bfill(axis=1)
        .iloc[:, 0]
    )

In [19]:
df_pres2017_lfn["code_commune"] = (
    df_pres2017_lfn["Code du département"].astype(str).str.zfill(2)
    + df_pres2017_lfn["Code de la commune"].astype(str).str.zfill(3)
)

df_pres2017_lfn["score_2017"] = 100*df_pres2017_lfn["voix"] / df_pres2017_lfn["Exprimés"]

In [20]:
df_pres2022_to_merge = df_pres2022[["code_commune", "Libellé de la commune", "score_2022"]]
df_pres2022_to_merge["libelle_commune_2022"] = df_pres2022_to_merge["Libellé de la commune"]
df_pres2022_to_merge = df_pres2022_to_merge.drop(columns=["Libellé de la commune"])

df_pres2017_to_merge = df_pres2017_lfn[["code_commune", "Libellé de la commune", "score_2017"]]
df_pres2017_to_merge["libelle_commune_2017"] = df_pres2017_to_merge["Libellé de la commune"]
df_pres2017_to_merge = df_pres2017_to_merge.drop(columns=["Libellé de la commune"])

df_pres2012_to_merge = df_pres2012[["code_commune", "Libellé de la commune", "score_2012"]]
df_pres2012_to_merge["libelle_commune_2012"] = df_pres2012_to_merge["Libellé de la commune"]
df_pres2012_to_merge = df_pres2012_to_merge.drop(columns=["Libellé de la commune"])

df_pres2007_to_merge = df_pres2007[["code_commune", "Libellé de la commune", "score_2007"]]
df_pres2007_to_merge["libelle_commune_2007"] = df_pres2007_to_merge["Libellé de la commune"]
df_pres2007_to_merge = df_pres2007_to_merge.drop(columns=["Libellé de la commune"])

In [21]:
df_pres = (
    df_pres2007_to_merge
    .merge(df_pres2012_to_merge, on="code_commune", how="outer")
    .merge(df_pres2017_to_merge, on="code_commune", how="outer")
    .merge(df_pres2022_to_merge, on="code_commune", how="outer")
)

In [22]:
df_long_pres = df_pres.melt(
    id_vars="code_commune",
    value_vars=[
        "score_2007",
        "score_2012",
        "score_2017",
        "score_2022"
    ],
    var_name="annee",
    value_name="score"
)

# Garder uniquement l'année
df_long_pres["annee"] = df_long_pres["annee"].str.extract(r"(\d{4})").astype(int)

# Trier
df_long_pres = df_long_pres.sort_values(["code_commune", "annee"]).reset_index(drop=True)

# EUROPEENNES

In [23]:
# Le jeu de données avec les résultats par candidat par bureau de vote
df_euro2024 = pd.read_excel("/Users/a33630/Documents/cinema/euro2024.xlsx")
df_euro2019 = pd.read_excel("/Users/a33630/Documents/cinema/euro2019.xls", engine='xlrd')
df_euro2014 = pd.read_excel("/Users/a33630/Documents/cinema/euro2014.xlsx")
df_euro2009_1 = pd.read_excel("/Users/a33630/Documents/cinema/euro2009_1.xls", engine='xlrd')
df_euro2009_2 = pd.read_excel("/Users/a33630/Documents/cinema/euro2009_2.xls", engine='xlrd')
df_euro2009 = pd.concat([df_euro2009_1, df_euro2009_2], ignore_index=True)
df_euro2004 = pd.read_excel("/Users/a33630/Documents/cinema/euro2004.xls", engine='xlrd')

In [24]:
cols_voix = [
    f"Voix {i}"
    for i in range(1, 39)
    if f"Voix {i}" in df_euro2024.columns
]

df_euro2024["voix"] = df_euro2024[cols_voix].sum(axis=1)
df_euro2024["score_2024"] = 100*df_euro2024["voix"] / df_euro2024["Exprimés"]
df_euro2024["code_commune"] = df_euro2024["Code commune"]

In [25]:
df_euro2019["code_commune"] = (
    df_euro2019["Code du département"].astype(str).str.zfill(2)
    + df_euro2019["Code de la commune"].astype(str).str.zfill(3)
)

In [26]:
df_euro2019["score_2019"] = 100*df_euro2019["voix"] / df_euro2019["Exprimés"]

In [27]:
# Colonnes correspondant aux listes 1 à 29
cols = []

for i in range(1, 30):
    cols.extend([
        f"N°Liste.{i}",
        f"Nuance Liste.{i}",
        f"Libellé Abrégé Liste.{i}",
        f"Nom Tête de Liste.{i}",
        f"Voix.{i}"
    ])

# Créer une copie
df_euro2014_lfn = df_euro2014.copy()

# Pour chaque liste : colonne sans suffixe + .1 à .24
for i in range(25):

    suffix = "" if i == 0 else f".{i}"

    # Garder uniquement les lignes où la nuance est LFN
    mask = df_euro2014_lfn[f"Nuance Liste{suffix}"] != "LFN"

    cols_a_caviarder = [
        f"N°Liste{suffix}",
        f"Nuance Liste{suffix}",
        f"Libellé Abrégé Liste{suffix}",
        f"Nom Tête de Liste{suffix}",
        f"Voix{suffix}"
    ]

    df_euro2014_lfn.loc[mask, cols_a_caviarder] = pd.NA

In [28]:
for nouvelle_colonne, prefixe in {
    "n_liste": "N°Liste",
    "nuance": "Nuance Liste",
    "libelle_liste": "Libellé Abrégé Liste",
    "tete_liste": "Nom Tête de Liste",
    "voix": "Voix"
}.items():
    
    cols = [f"{prefixe}.{i}" for i in range(1, 30)]
    
    df_euro2014_lfn[nouvelle_colonne] = (
        df_euro2014_lfn[cols]
        .bfill(axis=1)
        .iloc[:, 0]
    )

In [29]:
df_euro2014_lfn["score_2014"] = 100*df_euro2014_lfn["voix"] / df_euro2014_lfn["Exprimés"]

In [30]:
df_euro2009["code_commune"] = (
    df_euro2009["Code du département"].astype(str).str.zfill(2)
    + df_euro2009["Code de la commune"].astype(str).str.zfill(3)
)

/var/folders/r9/cjr0gj197xd220l1s09cyl_c0000gp/T/ipykernel_79709/87369958.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_euro2009["code_commune"] = (


In [31]:
# Créer une copie
df_euro2009_lfn = df_euro2009.copy()

# Pour chaque liste : colonne sans suffixe + .1 à .24
for i in range(25):

    suffix = "" if i == 0 else f".{i}"

    # Garder uniquement les lignes où la nuance est LFN
    mask = df_euro2009_lfn[f"Nuance Liste{suffix}"] != "LFN"

    cols_a_caviarder = [
        f"N°Liste{suffix}",
        f"Nuance Liste{suffix}",
        f"Libellé Abrégé Liste{suffix}",
        f"Nom Tête de Liste{suffix}",
        f"Voix{suffix}"
    ]

    df_euro2009_lfn.loc[mask, cols_a_caviarder] = pd.NA

In [32]:
for nouvelle_colonne, prefixe in {
    "n_liste": "N°Liste",
    "nuance": "Nuance Liste",
    "libelle_liste": "Libellé Abrégé Liste",
    "tete_liste": "Nom Tête de Liste",
    "voix": "Voix"
}.items():
    
    cols = [f"{prefixe}.{i}" for i in range(1, 25)]
    
    df_euro2009_lfn[nouvelle_colonne] = (
        df_euro2009_lfn[cols]
        .bfill(axis=1)
        .iloc[:, 0]
    )

In [33]:
df_euro2009_lfn["score_2009"] = 100*df_euro2009_lfn["voix"] / df_euro2009_lfn["Exprimés"]

In [34]:
df_euro2004["code_commune"] = (
    df_euro2004["Code du département"].astype(str).str.zfill(2)
    + df_euro2004["Code de la commune"].astype(str).str.zfill(3)
)

In [35]:
# Créer une copie
df_euro2004_lfn = df_euro2004.copy()

# Pour chaque liste, conserver les informations uniquement si la nuance est LFN
# Pour chaque liste : colonne sans suffixe + .1 à .24
for i in range(25):

    suffix = "" if i == 0 else f".{i}"

    # Garder uniquement les lignes où la nuance est LFN
    mask = df_euro2004_lfn[f"Nuance Liste{suffix}"] != "LFN"

    cols_a_caviarder = [
        f"N°Liste{suffix}",
        f"Nuance Liste{suffix}",
        f"Libellé Abrégé Liste{suffix}",
        f"Nom Tête de Liste{suffix}",
        f"Voix{suffix}"
    ]

    df_euro2004_lfn.loc[mask, cols_a_caviarder] = pd.NA

In [36]:
for nouvelle_colonne, prefixe in {
    "n_liste": "N°Liste",
    "nuance": "Nuance Liste",
    "libelle_liste": "Libellé Abrégé Liste",
    "tete_liste": "Nom Tête de Liste",
    "voix": "Voix"
}.items():
    
    cols = [f"{prefixe}.{i}" for i in range(1, 25)]
    
    df_euro2004_lfn[nouvelle_colonne] = (
        df_euro2004_lfn[cols]
        .bfill(axis=1)
        .iloc[:, 0]
    )

In [37]:
df_euro2004_lfn["score_2004"] = 100*df_euro2004_lfn["voix"] / df_euro2004_lfn["Exprimés"]
df_euro2004_lfn[["code_commune", "n_liste", "nuance", "libelle_liste", "tete_liste", "voix", "score_2004"]].head()

,code_commune,n_liste,nuance,libelle_liste,tete_liste,voix,score_2004
0,01001,<NA>,LFN,Liste du Front national,LE PEN,25.0,11.574074
1,01002,<NA>,LFN,Liste du Front national,LE PEN,15.0,15.957447
2,01004,<NA>,LFN,Liste du Front national,LE PEN,356.0,12.842713
3,01005,<NA>,LFN,Liste du Front national,LE PEN,58.0,18.589744
4,01006,<NA>,LFN,Liste du Front national,LE PEN,12.0,21.052632


In [38]:
df_euro2004_to_merge = df_euro2004_lfn[["code_commune", "Libellé de la commune", "score_2004"]]
df_euro2004_to_merge["libelle_commune_2004"] = df_euro2004_to_merge["Libellé de la commune"]
df_euro2004_to_merge = df_euro2004_to_merge.drop(columns=["Libellé de la commune"])

df_euro2009_to_merge = df_euro2009_lfn[["code_commune", "Libellé de la commune", "score_2009"]]
df_euro2009_to_merge["libelle_commune_2009"] = df_euro2009_to_merge["Libellé de la commune"]
df_euro2009_to_merge = df_euro2009_to_merge.drop(columns=["Libellé de la commune"])

df_euro2014_to_merge = df_euro2014_lfn[["code_commune", "Libellé de la commune", "score_2014"]]
df_euro2014_to_merge["libelle_commune_2014"] = df_euro2014_to_merge["Libellé de la commune"]
df_euro2014_to_merge = df_euro2014_to_merge.drop(columns=["Libellé de la commune"])

df_euro2019_to_merge = df_euro2019[["code_commune", "Libellé de la commune", "score_2019"]]
df_euro2019_to_merge["libelle_commune_2019"] = df_euro2019_to_merge["Libellé de la commune"]
df_euro2019_to_merge = df_euro2019_to_merge.drop(columns=["Libellé de la commune"])

df_euro2024_to_merge = df_euro2024[["code_commune", "Libellé commune", "score_2024"]]
df_euro2024_to_merge["libelle_commune_2024"] = df_euro2024_to_merge["Libellé commune"]
df_euro2024_to_merge = df_euro2024_to_merge.drop(columns=["Libellé commune"])

In [39]:
df = (
    df_euro2004_to_merge
    .merge(df_euro2009_to_merge, on="code_commune", how="outer")
    .merge(df_euro2014_to_merge, on="code_commune", how="outer")
    .merge(df_euro2019_to_merge, on="code_commune", how="outer")
    .merge(df_euro2024_to_merge, on="code_commune", how="outer")
)

In [40]:
df.isnull().sum()

code_commune               0
score_2004              1666
libelle_commune_2004     385
score_2009              1799
libelle_commune_2009     303
score_2014               319
libelle_commune_2014     319
score_2019              2035
libelle_commune_2019    2035
score_2024              2069
libelle_commune_2024    2067
dtype: int64

il y a des communes qui changent donc c'est pour ça qu'on ne retrouve pas les mêmes codes. A voir par la suite, si ça nous aide.

In [41]:
df_long = df.melt(
    id_vars="code_commune",
    value_vars=[
        "score_2004",
        "score_2009",
        "score_2014",
        "score_2019",
        "score_2024"
    ],
    var_name="annee",
    value_name="score"
)

# Garder uniquement l'année
df_long["annee"] = df_long["annee"].str.extract(r"(\d{4})").astype(int)

# Trier
df_long = df_long.sort_values(["code_commune", "annee"]).reset_index(drop=True)

In [42]:
df_long.isna().sum()

code_commune       0
annee              0
score           7888
dtype: int64

# Merge cinema et commune

Le dataframe a l'air de ressembler à quelque chose de plaisant donc il faut continuer avec présidentielles et législatives

In [65]:
df_elections = pd.concat(
    [df_long, df_long_pres],
    ignore_index=True
)

df_final = df_elections.merge(
    df_cinema_agg,
    on=["code_commune", "annee"],
    how="outer"
)

In [66]:
df_final.head(20)

,code_commune,annee,score,ecrans,fauteuils,nb_cinemas,evol
0,01001,2004,11.574074,NaN,NaN,NaN,NaN
1,01001,2007,11.877395,NaN,NaN,NaN,NaN
2,01001,2009,6.726457,NaN,NaN,NaN,NaN
3,01001,2012,25.250501,NaN,NaN,NaN,NaN
4,01001,2014,29.389313,NaN,NaN,NaN,NaN
5,01001,2017,25.454545,NaN,NaN,NaN,NaN
6,01001,2019,24.840764,NaN,NaN,NaN,NaN
7,01001,2022,28.653846,NaN,NaN,NaN,NaN
8,01001,2024,42.818428,NaN,NaN,NaN,NaN
9,01002,2004,15.957447,NaN,NaN,NaN,NaN


In [67]:
df_final["nb_cinemas"] = df_final["nb_cinemas"].fillna(0)

# ANALYSE

Il faut prendre en compte les communes où il n'y a pas de cinéma. Pq pour la commune 01004 il n'y a pas de cinéma mais des écrans et fauteuils.

In [68]:
df_reg = df_final.dropna(subset=["score"]).reset_index(drop=True)

In [69]:
df_reg.head()

,code_commune,annee,score,ecrans,fauteuils,nb_cinemas,evol
0,01001,2004,11.574074,NaN,NaN,0.0,NaN
1,01001,2007,11.877395,NaN,NaN,0.0,NaN
2,01001,2009,6.726457,NaN,NaN,0.0,NaN
3,01001,2012,25.250501,NaN,NaN,0.0,NaN
4,01001,2014,29.389313,NaN,NaN,0.0,NaN


In [70]:
len(df_reg)

321941

In [72]:
df_reg = df_reg.set_index(["code_commune", "annee"])

In [74]:
modele = PanelOLS(
    df_reg["score"],
    df_reg[["nb_cinemas"]],
    entity_effects=True,
    time_effects=True
)

resultats = modele.fit()

print(resultats)

                          PanelOLS Estimation Summary                           
Dep. Variable:                  score   R-squared:                        0.0014
Estimator:                   PanelOLS   R-squared (Between):              0.0045
No. Observations:              321941   R-squared (Within):              -0.0023
Date:                Thu, Aug 20 2026   R-squared (Overall):              0.0034
Time:                        16:41:41   Log-likelihood                -9.515e+05
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      400.39
Entities:                       37302   P-value                           0.0000
Avg Obs:                       8.6307   Distribution:                F(1,284630)
Min Obs:                       1.0000                                           
Max Obs:                       9.0000   F-statistic (robust):             400.39
                            

J'ai identifié les cinémas qui ont fermé sur la période (643), maintenant il faut pouvoir identifier les communes qui ont été affectées par cela (10km à la ronde). Puis lancer l'analyse avec les contrefactuels (observations précédentes et après)